# Dataset Creation Human

This notebook prepares the input dataset required by the UTTOPIA Random Forest model. It integrates mRNA abundance, co-expression, STRING interaction scores, and subcellular localisation for gene pairs.

## Input format

Three input formats are supported. Select the one that matches your data:

| Table | Columns | Use when |
|-------|---------|----------|
| **A** | `gene_A, gene_B, mRNA_A, mRNA_B, coex` | You have your own pre-calculated co-expression values |
| **B** | `gene_A, gene_B, mRNA_A, mRNA_B` | You have specific gene pairs; co-expression is computed from the default database |
| **C** | `gene, mRNA` | You have a gene list; all pairwise combinations are generated automatically |

## Imports

In [ ]:
import pandas as pd
import numpy as np
import os
import pearsonr
from huggingface_hub import hf_hub_download

## Load reference datasets

In [ ]:
HF_REPO = "caioo61/UTTOPIA-RF-ESM"
BASE_PATH = os.getcwd()

def download_from_hf(filename):
    """ Download from Hugging Face """
    return hf_hub_download(
        repo_id = HF_REPO,
        filename = filename,
        repo_type = "dataset",
        local_dir = BASE_PATH,
        local_dir_use_symlinks = False
    )

In [ ]:
path_string = f"Dataset/Human/String_Human.csv"
path_coex   = f"Dataset/Human/Coexpression_Human.csv"
path_loca   = f"Dataset/Human/Localisation_Human.csv"

string = pd.read_csv(download_from_hf(path_string), index_col = 0)
coex = pd.read_csv(download_from_hf(path_coex), index_col = 0)
localisation = pd.read_csv(download_from_hf(path_loca), index_col = 0)

## Helper functions

In [ ]:
def get_coex(mRNA, coex):
    """ Compute Pearson co-expression between each gene pair using the reference expression matrix """
    orfs = list(np.unique(list(mRNA['Cmin']) + list(mRNA['Cmax'])))
    sample = coex[coex['symbol'].isin(orfs)]

    # Build a symbol → expression vector lookup
    expr_dict = dict(zip(
        sample['symbol'],
        sample.iloc[:, 1:].values
    ))

    coex_list = []

    for i, (p1, p2) in enumerate(zip(mRNA['Cmin'], mRNA['Cmax'])):
        if p1 in expr_dict and p2 in expr_dict:
            x = expr_dict[p1]
            y = expr_dict[p2]
            # Keep only positions where both genes have valid expression values
            mask = (~np.isnan(x)) & (~np.isnan(y))

            if np.sum(mask) >= 2:
                r = pearsonr(x[mask], y[mask]).statistic
            else:
                r = np.nan

            coex_list.append(r)
        else:
            coex_list.append(np.nan)

    mRNA['coex'] = coex_list
    # Drop pairs where co-expression could not be computed
    mRNA_coex = mRNA.loc[~(pd.isna(mRNA['coex'])), :]
    mRNA_coex.index = range(mRNA_coex.shape[0])
    mRNA_coex['coex'] = np.abs(mRNA_coex['coex'])
    return mRNA_coex


def get_string(mRNA_coex, string):
    """ Left-join STRING interaction scores onto gene pairs """
    mRNA_coex_string = mRNA_coex.merge(string,
        left_on = ['Cmin', 'Cmax'],
        right_on = ['protein1', 'protein2'],
        how = 'left'
    ).drop(columns = ['protein1', 'protein2'])
    return mRNA_coex_string


def get_loca(mRNA_coex_string, localisation):
    """ Attach subcellular localisation scores for both genes in each pair (Cmin and Cmax) """
    loca_copy = localisation.copy()
    # Keep only pairs where both genes have localisation data
    mRNA_coex_string_loca = mRNA_coex_string[
        mRNA_coex_string['Cmin'].isin(list(loca_copy['ORF']))
    ]
    mRNA_coex_string_loca = mRNA_coex_string_loca[
        mRNA_coex_string_loca['Cmax'].isin(list(loca_copy['ORF']))
    ]

    # Prefix columns with 'Cmin_' and merge localisation for the lower-expressed gene
    name = ['Cmin_' + loca_copy.columns[i] for i in range(1, loca_copy.shape[1])]
    loca_copy.columns = ['ORF'] + name

    mRNA_coex_string_loca1 = mRNA_coex_string_loca.merge(
        loca_copy,
        left_on = 'Cmin',
        right_on = 'ORF',
        how = 'left'
    ).drop(columns = ['ORF'])

    # Prefix columns with 'Cmax_' and merge localisation for the higher-expressed gene
    loca_copy.columns = localisation.columns
    name = ['Cmax_' + loca_copy.columns[i] for i in range(1, loca_copy.shape[1])]
    loca_copy.columns = ['ORF'] + name

    mRNA_coex_string_loca = mRNA_coex_string_loca1.merge(
        loca_copy,
        left_on = 'Cmax',
        right_on = 'ORF',
        how = 'left'
    ).drop(columns = ['ORF'])

    return mRNA_coex_string_loca


def generate_pairs(mRNA):
    """ Generate all pairwise combinations from a gene list (Table C input) """
    gene_A = []
    gene_B = []
    print(mRNA.shape)
    for i in range(mRNA.shape[0]):
        for j in range(mRNA.shape[0]):
            if mRNA['gene'][i] == mRNA['gene'][j]: continue
            gene_A.append(mRNA['gene'][i])
            gene_B.append(mRNA['gene'][j])

    pairs = pd.DataFrame({'gene_A': gene_A, 'gene_B': gene_B})

    mRNA.columns = ['gene', 'mRNA_A']
    pairs_mRNA = pairs.merge(mRNA, left_on='gene_A', right_on='gene', how='left').drop(columns='gene')

    mRNA.columns = ['gene', 'mRNA_B']
    pairs_mRNA = pairs_mRNA.merge(mRNA, left_on='gene_B', right_on='gene', how='left').drop(columns='gene')

    return pairs_mRNA

## Load input file and detect format

In [26]:
input_file_path = 'example_human/example_input_file.csv'

In [27]:
input_file = pd.read_csv(input_file_path, index_col=0)

## Rearrange gene pairs

Genes are sorted so that `Cmin` always holds the gene with lower mRNA abundance and `Cmax` the one with higher abundance. This ordering is required by the Random Forest model.

In [28]:
# Detect input table type based on column structure
if len(input_file.columns) == 5 and list(input_file.columns) == ['gene_A', 'gene_B', 'mRNA_A', 'mRNA_B', 'coex']:
    option = 'A'
    coex_values = input_file['coex']
    input_file = input_file.drop(columns=['coex'])

elif len(input_file.columns) == 4 and list(input_file.columns) == ['gene_A', 'gene_B', 'mRNA_A', 'mRNA_B']:
    option = 'B'

elif len(input_file.columns) == 2 and list(input_file.columns) == ['gene', 'mRNA']:
    option = 'C'
    input_file = generate_pairs(input_file)

else:
    print('Column names need to be corrected')

print(option)

B


In [29]:
# Swap gene_A / gene_B so that mRNA_A ≤ mRNA_B in every row
mask = input_file['mRNA_A'] > input_file['mRNA_B']
cols1 = ['gene_A', 'mRNA_A']
cols2 = ['gene_B', 'mRNA_B']

copy = input_file.copy()
tmp = copy.loc[mask, cols1].values
copy.loc[mask, cols1] = copy.loc[mask, cols2].values
copy.loc[mask, cols2] = tmp
mRNA_rearranged = copy
mRNA_rearranged.columns = ['Cmin', 'Cmax', 'Cmin_rna', 'Cmax_rna']

## Integrate STRING, co-expression, and localisation

In [30]:
# Add STRING interaction scores
mRNA_string = get_string(mRNA_rearranged, string)

# Add co-expression: use user-provided values (option A) or compute from the database
if option != 'A':
    mRNA_coex_string = get_coex(mRNA_string, coex)
else:
    mRNA_coex_string = mRNA_string
    mRNA_coex_string['coex'] = coex_values

# Reorder columns to match the expected RF model input
mRNA_coex_string = mRNA_coex_string[['Cmin', 'Cmax', 'Cmin_rna', 'Cmax_rna', 'coex', 'string']]

# Add subcellular localisation features for both genes
mRNA_coex_string_loca = get_loca(mRNA_coex_string, localisation)

## Export

In [ ]:
# Save the final feature matrix — ready to feed into the RF model
mRNA_coex_string_loca.to_csv(os.path.join(BASE_PATH, "input_to_RF_Human.csv"), index=False)